In [1]:
from pydantic import BaseModel
from openai import OpenAI
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from src.rarekg.pubmed.pubmed_central import get_pmc_fulltext
from src.rarekg.sgr import (extract_entities, ExtractionResult)


pmcid = "PMC9314610"

title, abstract, para = get_pmc_fulltext(pmcid)
print(title)
print(abstract)







Genotype–phenotype correlates in Joubert syndrome: A review
Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic variants in TMEM67 have a significantly higher risk of liver fibrosis, while pathogenic variants in NPHP1 , RPGRIP1L , and TMEM237 are frequently associated to JS with renal involvement, requiring a closer monitoring of liver parameters, or renal functioning. On the other hand, individuals with causal variants in the CEP290 or AHI1 need a closer surveillance for retinal dystrophy and, in case of CEP290 , also for chronic kidney disease. These

In [4]:
from typing import Set, Tuple, Union, Dict, Any
import json


def _normalize_entity_name(name: str) -> str:
    """
    Very simple text normalization for comparison:
      - lowercase
      - strip leading/trailing whitespace
      - collapse internal whitespace

    You can later extend this (e.g. Greek letters, hyphens, punctuation).
    """
    name = name.strip().lower()
    # collapse multiple spaces/tabs/newlines
    parts = name.split()
    return " ".join(parts)


def _entities_from_json_like(
    data: Union[str, Dict[str, Any], ExtractionResult]
) -> Set[Tuple[str, str]]:
   
    if isinstance(data, ExtractionResult):
        ents = data.entities
        result: Set[Tuple[str, str]] = set()
        for e in ents:
            if hasattr(e.type, "value"):
                etype = e.type.value
            else:
                etype = str(e.type)
                # if it's something like "EntityType.gene", strip prefix
                if "." in etype:
                    etype = etype.split(".")[-1]

            if etype == "gene":
                name = e.name              # no normalization
            else:
                name = _normalize_entity_name(e.name)
            result.add((name, etype))
        return result
    
    if isinstance(data, str):
        data = json.loads(data)

    entities = data.get("entities", [])
    result: Set[Tuple[str, str]] = set()
    for ent in entities:
        name = ent.get("name")
        etype = ent.get("type")
        if not name or not etype:
            continue
        if etype == "gene":
            norm_name = name  # keep as is
        else:
            norm_name = _normalize_entity_name(name)

        result.add((_normalize_entity_name(norm_name), str(etype)))
    return result


def evaluate_extraction(
    gold: Union[str, Dict[str, Any], ExtractionResult],
    pred: Union[str, Dict[str, Any], ExtractionResult],
) -> Dict[str, float]:
    gold_set = _entities_from_json_like(gold)
    pred_set = _entities_from_json_like(pred)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    num_gold = len(gold_set)
    num_pred = len(pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "true_positives": float(tp),
        "false_positives": float(fp),
        "false_negatives": float(fn),
        "num_gold": float(num_gold),
        "num_pred": float(num_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


In [16]:
import json
from pathlib import Path

json_path = Path("../golden/abstract.json")

with json_path.open("r", encoding="utf-8") as f:
    gold_p1 = json.load(f)

print(gold_p1.keys())        # -> dict with "entities"
print(gold_p1["entities"][:2])
i=8
print(gold_p1["entities"][i]["type"] == "gene")
print(gold_p1["entities"][i]["type"])
ab = _entities_from_json_like(gold_p1)
print(ab)

dict_keys(['entities'])
[{'name': 'Joubert syndrome', 'type': 'rare_disease'}, {'name': 'cerebellar malformation', 'type': 'phenotype'}]
True
gene
{('retinal dystrophy', 'phenotype'), ('rpgrip1l', 'gene'), ('cep290', 'gene'), ('cerebellar malformation', 'phenotype'), ('tmem237', 'gene'), ('ahi1', 'gene'), ('molar tooth sign', 'phenotype'), ('brainstem malformation', 'phenotype'), ('chronic kidney disease', 'phenotype'), ('joubert syndrome', 'rare_disease'), ('tmem67', 'gene'), ('liver fibrosis', 'phenotype'), ('nphp1', 'gene')}


In [26]:
print(len(para))

68


In [29]:
print(para[67])

The establishment of meaningful genotype–phenotype correlates, which pertains not only to JS and other primary ciliopathies but to the majority of inherited diseases, represents the greatest challenge of genetic research for the years to come; deeper phenotyping of patients, analysis of large cohorts through multicenter collaborations and a better understanding of our genomic structure and variability at the individual level will represent essential assets to successfully accomplish this task.


In [2]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

In [3]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

result = extract_entities(
    abstract,
    client,
    model="medgemma",
    entity_group="all",          # or "rare_disease", "phenotype", "treatment"
    use_few_shot=True,
    use_response_format=False,   # vLLM: use guided_json
)

print(result)
for e in result.entities:
    print(e.name, e.type)


entities=[Entity(name='Joubert syndrome', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='ciliopathy', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='cerebellar malformation', type=<EntityType.phenotype: 'phenotype'>), Entity(name='brainstem malformation', type=<EntityType.phenotype: 'phenotype'>), Entity(name='molar tooth sign', type=<EntityType.phenotype: 'phenotype'>), Entity(name='TMEM67', type=<EntityType.gene: 'gene'>), Entity(name='NPHP1', type=<EntityType.gene: 'gene'>), Entity(name='RPGRIP1L', type=<EntityType.gene: 'gene'>), Entity(name='TMEM237', type=<EntityType.gene: 'gene'>), Entity(name='CEP290', type=<EntityType.gene: 'gene'>), Entity(name='AHI1', type=<EntityType.gene: 'gene'>), Entity(name='liver fibrosis', type=<EntityType.phenotype: 'phenotype'>), Entity(name='renal involvement', type=<EntityType.phenotype: 'phenotype'>), Entity(name='retinal dystrophy', type=<EntityType.phenotype: 'phenotype'>), Entity(name='chronic kidney disease', typ

In [8]:
result = extract_entities(
    abstract,
    client,
    model="medgemma",
    entity_group="rare_disease",          # or "rare_disease", "phenotype", "treatment"
    use_few_shot=True,
    use_response_format=False,   # vLLM: use guided_json
)

print(result)
for e in result.entities:
    print(e.name, e.type)


entities=[Entity(name='Joubert syndrome', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='ciliopathy', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='cerebellar malformation', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='brainstem malformation', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='molar tooth sign', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='TMEM67', type=<EntityType.gene: 'gene'>), Entity(name='NPHP1', type=<EntityType.gene: 'gene'>), Entity(name='RPGRIP1L', type=<EntityType.gene: 'gene'>), Entity(name='TMEM237', type=<EntityType.gene: 'gene'>), Entity(name='CEP290', type=<EntityType.gene: 'gene'>), Entity(name='AHI1', type=<EntityType.gene: 'gene'>), Entity(name='liver fibrosis', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='renal involvement', type=<EntityType.rare_disease: 'rare_disease'>), Entity(name='retinal dystrophy', type=<EntityType.rare_disease: 'rare_disease'>), Entit